# Human Ratings vs Claude Scores — Comparison Analysis
**Dataset:** 800 images (400 urban, 400 landscape)  
- **Claude scores:** 15 aesthetic/architectural properties (1–10 each)  
- **Human scores:** Crowd-sourced scenic quality ratings (1–10 average)

## 1. Setup & Data Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats as scipy_stats
import warnings
warnings.filterwarnings('ignore')

# ── Load datasets ──────────────────────────────────────────────────
claude_df = pd.read_csv('all_images_15_properties.csv')
human_df  = pd.read_csv('votes__1_.tsv', sep='\t')

# ── Build join key ─────────────────────────────────────────────────
claude_df['ID'] = claude_df['image_name'].str.replace('.jpg', '', regex=False).astype(int)

# ── Merge ──────────────────────────────────────────────────────────
df = claude_df.merge(human_df[['ID', 'Average', 'Variance', 'Votes']], on='ID')
df.rename(columns={'Average': 'human_score', 'Variance': 'human_variance'}, inplace=True)

# ── Properties list ────────────────────────────────────────────────
properties = [
    'levels_of_scale', 'strong_centers', 'boundaries', 'alternating_repetition',
    'positive_space', 'good_shape', 'local_symmetries', 'deep_interlock_and_ambiguity',
    'contrast', 'gradients', 'roughness', 'echoes', 'the_void',
    'simplicity_and_inner_calm', 'not_separateness'
]
pretty = [
    'Levels of Scale', 'Strong Centers', 'Boundaries', 'Alternating Repetition',
    'Positive Space', 'Good Shape', 'Local Symmetries', 'Deep Interlock & Ambiguity',
    'Contrast', 'Gradients', 'Roughness', 'Echoes', 'The Void',
    'Simplicity & Inner Calm', 'Not-Separateness'
]
prop_label = dict(zip(properties, pretty))

# ── Composite Claude score ─────────────────────────────────────────
df['claude_mean'] = df[properties].mean(axis=1)

# ── Colours ───────────────────────────────────────────────────────
C_URBAN = '#E74C3C'
C_LAND  = '#2980B9'
C_MAP   = {'urban': C_URBAN, 'landscape': C_LAND}

print(f"Merged rows : {len(df)}")
print(f"Image types : {df['image_type'].value_counts().to_dict()}")
print(f"Human score : mean={df['human_score'].mean():.2f}, std={df['human_score'].std():.2f}")
print(f"Claude mean : mean={df['claude_mean'].mean():.2f}, std={df['claude_mean'].std():.2f}")
df[['image_name','image_type','claude_mean','human_score','human_variance']].head(8)

## 2. Global Correlation — Claude Mean vs Human Score

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, itype in zip(axes, ['urban', 'landscape']):
    sub = df[df['image_type'] == itype]
    color = C_MAP[itype]
    r, p   = scipy_stats.pearsonr(sub['claude_mean'], sub['human_score'])
    rho, _ = scipy_stats.spearmanr(sub['claude_mean'], sub['human_score'])

    ax.scatter(sub['claude_mean'], sub['human_score'],
               color=color, alpha=0.45, s=30, edgecolors='none')

    # Regression line
    m, b = np.polyfit(sub['claude_mean'], sub['human_score'], 1)
    xs = np.linspace(sub['claude_mean'].min(), sub['claude_mean'].max(), 100)
    ax.plot(xs, m*xs + b, color='black', linewidth=2, linestyle='--')

    ax.set_xlabel('Claude Mean Score', fontsize=12)
    ax.set_ylabel('Human Rating (avg)', fontsize=12)
    ax.set_title(f'{itype.capitalize()}\nPearson r={r:.3f}  |  Spearman ρ={rho:.3f}  |  p={p:.2e}',
                 fontsize=12, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Claude Mean Score vs Human Rating', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('c01_scatter_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# Overall stats
r_all, p_all     = scipy_stats.pearsonr(df['claude_mean'], df['human_score'])
rho_all, p_all2  = scipy_stats.spearmanr(df['claude_mean'], df['human_score'])
print(f"Overall  Pearson  r  = {r_all:.4f}  (p={p_all:.2e})")
print(f"Overall  Spearman ρ  = {rho_all:.4f}  (p={p_all2:.2e})")

## 3. Per-Property Correlation with Human Score

In [ ]:
rows = []
for prop, label in zip(properties, pretty):
    for itype in ['urban', 'landscape', 'all']:
        sub = df if itype == 'all' else df[df['image_type'] == itype]
        r, p = scipy_stats.pearsonr(sub[prop], sub['human_score'])
        rows.append({'Property': label, 'Type': itype, 'Pearson_r': r, 'p_value': p})

corr_df = pd.DataFrame(rows)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
for ax, itype, color in zip(axes,
                              ['urban', 'landscape', 'all'],
                              [C_URBAN, C_LAND, '#7F8C8D']):
    sub = corr_df[corr_df['Type'] == itype].sort_values('Pearson_r', ascending=True)
    colors = [color if p < 0.05 else '#CCCCCC' for p in sub['p_value']]
    ax.barh(sub['Property'], sub['Pearson_r'], color=colors, edgecolor='white', alpha=0.9)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Pearson r', fontsize=11)
    ax.set_title(f'{itype.capitalize()}\n(grey = p≥0.05)', fontsize=12, fontweight='bold')
    ax.grid(axis='x', linestyle='--', alpha=0.4)
    ax.set_xlim(-0.5, 0.7)

plt.suptitle('Per-Property Correlation with Human Score', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('c02_per_property_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Score Distribution Comparison (Overlapping Histograms)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, itype in zip(axes, ['urban', 'landscape']):
    sub = df[df['image_type'] == itype]
    ax.hist(sub['human_score'], bins=30, alpha=0.65, color='#27AE60', label='Human', edgecolor='white')
    ax.hist(sub['claude_mean'], bins=30, alpha=0.65, color=C_MAP[itype], label='Claude Mean', edgecolor='white')
    ax.axvline(sub['human_score'].mean(), color='#27AE60', linestyle='--', linewidth=2)
    ax.axvline(sub['claude_mean'].mean(), color=C_MAP[itype], linestyle='--', linewidth=2)
    ax.set_xlabel('Score (1–10)', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title(f'{itype.capitalize()} — Score Distributions', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('c03_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Bland-Altman Plot (Agreement Analysis)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, itype in zip(axes, ['urban', 'landscape']):
    sub = df[df['image_type'] == itype].copy()
    sub['avg_score'] = (sub['claude_mean'] + sub['human_score']) / 2
    sub['diff']      =  sub['claude_mean'] - sub['human_score']

    mean_diff = sub['diff'].mean()
    std_diff  = sub['diff'].std()
    loa_upper = mean_diff + 1.96 * std_diff
    loa_lower = mean_diff - 1.96 * std_diff

    ax.scatter(sub['avg_score'], sub['diff'], alpha=0.4, s=25,
               color=C_MAP[itype], edgecolors='none')
    ax.axhline(mean_diff,  color='black',  linewidth=1.8, linestyle='-',  label=f'Mean diff = {mean_diff:.2f}')
    ax.axhline(loa_upper,  color='tomato', linewidth=1.5, linestyle='--', label=f'+1.96 SD = {loa_upper:.2f}')
    ax.axhline(loa_lower,  color='tomato', linewidth=1.5, linestyle='--', label=f'-1.96 SD = {loa_lower:.2f}')
    ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')

    ax.set_xlabel('Mean of Claude & Human Score', fontsize=11)
    ax.set_ylabel('Claude − Human', fontsize=11)
    ax.set_title(f'Bland-Altman: {itype.capitalize()}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.3)

plt.suptitle('Bland-Altman Agreement Plot (Claude Mean vs Human Rating)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('c04_bland_altman.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Rank Agreement — Concordance Plot (Top / Bottom Quartile)

In [ ]:
results = []
for itype in ['urban', 'landscape']:
    sub = df[df['image_type'] == itype].copy()
    q25_h, q75_h = sub['human_score'].quantile([0.25, 0.75])
    q25_c, q75_c = sub['claude_mean'].quantile([0.25, 0.75])

    sub['human_tier']  = pd.cut(sub['human_score'],
                                bins=[-np.inf, q25_h, q75_h, np.inf],
                                labels=['Bottom 25%', 'Middle 50%', 'Top 25%'])
    sub['claude_tier'] = pd.cut(sub['claude_mean'],
                                bins=[-np.inf, q25_c, q75_c, np.inf],
                                labels=['Bottom 25%', 'Middle 50%', 'Top 25%'])

    ct = pd.crosstab(sub['human_tier'], sub['claude_tier'],
                     rownames=['Human'], colnames=['Claude'])

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(ct.values, cmap='Blues', aspect='auto')
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(ct.columns, fontsize=10)
    ax.set_yticklabels(ct.index, fontsize=10)
    ax.set_xlabel('Claude Tier', fontsize=11)
    ax.set_ylabel('Human Tier', fontsize=11)
    ax.set_title(f'Rank Agreement — {itype.capitalize()}', fontsize=13, fontweight='bold')
    for i in range(3):
        for j in range(3):
            pct = ct.values[i, j] / ct.values.sum() * 100
            ax.text(j, i, f'{ct.values[i,j]}\n({pct:.1f}%)',
                    ha='center', va='center', fontsize=11, fontweight='bold',
                    color='white' if ct.values[i, j] > ct.values.max()*0.6 else 'black')
    plt.colorbar(im, ax=ax, label='Count')
    plt.tight_layout()
    plt.savefig(f'c05_rank_agreement_{itype}.png', dpi=150, bbox_inches='tight')
    plt.show()

    agree = (sub['human_tier'] == sub['claude_tier']).mean()
    print(f"{itype.capitalize()}: exact-tier agreement = {agree:.1%}")

## 7. Individual Property vs Human Score (Interactive Scatter)

In [ ]:
df_melted = df.melt(
    id_vars=['image_name', 'image_type', 'human_score'],
    value_vars=properties,
    var_name='property', value_name='claude_score'
)
df_melted['property_label'] = df_melted['property'].map(prop_label)

fig = px.scatter(
    df_melted,
    x='claude_score',
    y='human_score',
    color='image_type',
    facet_col='property_label',
    facet_col_wrap=5,
    opacity=0.35,
    color_discrete_map={'urban': C_URBAN, 'landscape': C_LAND},
    trendline='ols',
    labels={'claude_score': 'Claude Score', 'human_score': 'Human Rating'},
    title='Each Claude Property vs Human Score (with OLS trendline)',
    height=750, width=1200
)
fig.update_traces(marker_size=4)
fig.update_layout(showlegend=True)
fig.show()
fig.write_html('c06_property_scatter_facets.html')

## 8. Human Score Variance vs Claude-Human Disagreement

In [ ]:
df['abs_diff'] = (df['claude_mean'] - df['human_score']).abs()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, itype in zip(axes, ['urban', 'landscape']):
    sub = df[df['image_type'] == itype]
    r, p = scipy_stats.pearsonr(sub['human_variance'], sub['abs_diff'])
    ax.scatter(sub['human_variance'], sub['abs_diff'],
               alpha=0.4, s=25, color=C_MAP[itype], edgecolors='none')
    m, b = np.polyfit(sub['human_variance'], sub['abs_diff'], 1)
    xs = np.linspace(sub['human_variance'].min(), sub['human_variance'].max(), 100)
    ax.plot(xs, m*xs + b, 'k--', linewidth=2)
    ax.set_xlabel('Human Rating Variance', fontsize=11)
    ax.set_ylabel('|Claude − Human|', fontsize=11)
    ax.set_title(f'{itype.capitalize()}\nPearson r={r:.3f}  p={p:.3f}', fontsize=12, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.3)

plt.suptitle('Human Disagreement (Variance) vs Claude-Human Gap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('c07_variance_vs_disagreement.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Quartile Score Profiles (Radar) — Human Top vs Bottom

In [ ]:
for itype in ['urban', 'landscape']:
    sub = df[df['image_type'] == itype]
    q25 = sub['human_score'].quantile(0.25)
    q75 = sub['human_score'].quantile(0.75)

    top_vals = sub[sub['human_score'] >= q75][properties].mean().tolist()
    bot_vals = sub[sub['human_score'] <= q25][properties].mean().tolist()

    fig = go.Figure()
    for vals, name, color in [(top_vals, 'Top 25% Human', '#27AE60'),
                               (bot_vals, 'Bottom 25% Human', '#E74C3C')]:
        fig.add_trace(go.Scatterpolar(
            r=vals + [vals[0]],
            theta=pretty + [pretty[0]],
            fill='toself', name=name,
            line_color=color,
            fillcolor=color.replace('#', 'rgba(').replace('27AE60','39,174,96,0.2)').replace('E74C3C','231,76,60,0.2)')
        ))

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 10])),
        title=dict(text=f'{itype.capitalize()} — Claude Property Profile: Human Top vs Bottom Quartile',
                   x=0.5, font_size=14),
        width=650, height=560
    )
    fig.show()
    fig.write_html(f'c08_radar_{itype}.html')

## 10. Statistical Summary Table

In [ ]:
rows = []
for prop, label in zip(properties, pretty):
    for itype in ['urban', 'landscape', 'all']:
        sub = df if itype == 'all' else df[df['image_type'] == itype]
        r,  p1 = scipy_stats.pearsonr(sub[prop], sub['human_score'])
        rho, _ = scipy_stats.spearmanr(sub[prop], sub['human_score'])
        rows.append({'Property': label, 'Subset': itype,
                     'Pearson r': round(r, 3),
                     'Spearman ρ': round(rho, 3),
                     'p-value': round(p1, 4),
                     'Significant': '✓' if p1 < 0.05 else '✗'})

stats_df = pd.DataFrame(rows)

print("=== All images — ranked by Pearson r ===")
display(stats_df[stats_df['Subset'] == 'all']
        .drop(columns='Subset')
        .sort_values('Pearson r', ascending=False)
        .reset_index(drop=True)
        .style.background_gradient(subset=['Pearson r', 'Spearman ρ'], cmap='RdYlGn'))

print("\n=== Overall composite score ===")
for itype in ['urban', 'landscape', 'all']:
    sub = df if itype == 'all' else df[df['image_type'] == itype]
    r, p   = scipy_stats.pearsonr(sub['claude_mean'], sub['human_score'])
    rho, _ = scipy_stats.spearmanr(sub['claude_mean'], sub['human_score'])
    mae    = (sub['claude_mean'] - sub['human_score']).abs().mean()
    bias   = (sub['claude_mean'] - sub['human_score']).mean()
    print(f"  {itype:10s}  Pearson r={r:.3f}  Spearman ρ={rho:.3f}  MAE={mae:.3f}  Bias(Claude-Human)={bias:+.3f}")